# 4. Machine learning
Random Forest and LightGBM, each with own-history features and with all 24 features, plus two extensions: a Random Forest on log volatility and a HAR + Random Forest hybrid.

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

rv = pd.read_csv("rv.csv", index_col=0, parse_dates=True)
feats = pd.read_csv("features.csv", index_col=0, parse_dates=True)
lin = pd.read_csv("harx_forecasts.csv", index_col=0, header=[0, 1], parse_dates=True)

Load the data and the HAR/HARX forecasts for comparison.

In [6]:
def ml_forecast(model, y, X, start="2021", refit_every=4):
    X = X.shift(1)
    data = pd.concat([y.rename("y"), X], axis=1).dropna()
    test_weeks = data.loc[start:].index
    preds = []
    for i in range(0, len(test_weeks), refit_every):
        block = test_weeks[i:i + refit_every]
        train = data.loc[:block[0]].iloc[:-1]
        model.fit(train.drop(columns="y"), train["y"])
        preds.append(pd.Series(model.predict(data.loc[block].drop(columns="y")), index=block))
    return pd.concat(preds)

Walk-forward for ML models: refit every 4 weeks on all earlier data, then predict the next 4 weeks. Still no look-ahead; refitting less often keeps the run time reasonable.

In [7]:
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=0)
lgbm = LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=15, verbose=-1, random_state=0)

fc = {"RF_own": {}, "RF_all": {}, "LGBM_own": {}, "LGBM_all": {}}
for c in rv:
    own = feats[[f"{c}_rv", f"{c}_rv4", f"{c}_rv12"]]
    fc["RF_own"][c] = ml_forecast(rf, rv[c], own)
    fc["RF_all"][c] = ml_forecast(rf, rv[c], feats)
    fc["LGBM_own"][c] = ml_forecast(lgbm, rv[c], own)
    fc["LGBM_all"][c] = ml_forecast(lgbm, rv[c], feats)

fc = {k: pd.DataFrame(v) for k, v in fc.items()}

Fixed, conservative settings, not tuned on the test period. Each model is run with own history only (`_own`) and with all features (`_all`).

In [8]:
test = rv.loc["2021":]
naive = rv.shift(1).loc["2021":]
rmse = lambda f: np.sqrt(((test - f)**2).mean())

table = pd.DataFrame({"naive": rmse(naive), "HAR": rmse(lin["HAR"]), "HARX": rmse(lin["HARX"])})
for k, v in fc.items():
    table[k] = rmse(v)
table.round(4)

,naive,HAR,HARX,RF_own,RF_all,LGBM_own,LGBM_all
bitcoin,0.3591,0.2760,0.2764,0.2831,0.2787,0.3110,0.2993
oil,0.1923,0.1634,0.1532,0.1716,0.1725,0.2052,0.2085
gold,0.1010,0.0834,0.0773,0.0845,0.0797,0.0878,0.0833
silver,0.2607,0.2023,0.2029,0.2102,0.2077,0.2250,0.2184


No ML model beats the best linear model. Random Forest is clearly better than LightGBM on this small, noisy dataset. For the trees, adding all features helps for Bitcoin, gold and silver, unlike for linear regression.

In [9]:
pd.concat(fc, axis=1).to_csv("ml_forecasts.csv")

Save the ML forecasts.

In [10]:
import statsmodels.api as sm

def hybrid_forecast(model, y, X_lin, X_ml, start="2021", refit_every=4):
    lin = sm.add_constant(X_lin).shift(1).add_prefix("lin_")
    ml = X_ml.shift(1).add_prefix("ml_")
    data = pd.concat([y.rename("y"), lin, ml], axis=1).dropna()
    test_weeks = data.loc[start:].index
    preds = []
    for i in range(0, len(test_weeks), refit_every):
        block = test_weeks[i:i + refit_every]
        train = data.loc[:block[0]].iloc[:-1]
        ols = sm.OLS(train["y"], train[lin.columns]).fit()
        model.fit(train[ml.columns], train["y"] - ols.fittedvalues)
        preds.append(ols.predict(data.loc[block, lin.columns]) + model.predict(data.loc[block, ml.columns]))
    return pd.concat(preds)

**Hybrid:** HARX makes the forecast and a Random Forest predicts HARX's error from all features. Motivated by a known weakness of trees: they cannot predict values beyond the range seen in training.

In [11]:
iv = {"oil": "ovx", "gold": "gvz", "silver": "gvz", "bitcoin": "vix"}

hyb, rflog = {}, {}
for c in rv:
    own = [f"{c}_rv", f"{c}_rv4", f"{c}_rv12"]
    hyb[c] = hybrid_forecast(rf, rv[c], feats[own + [iv[c]]], feats)
    rflog[c] = np.exp(ml_forecast(rf, np.log(rv[c]), feats))

hyb, rflog = pd.DataFrame(hyb), pd.DataFrame(rflog)

table["RF_log"] = rmse(rflog)
table["Hybrid"] = rmse(hyb)
table.round(4)

,naive,HAR,HARX,RF_own,RF_all,LGBM_own,LGBM_all,RF_log,Hybrid
bitcoin,0.3591,0.2760,0.2764,0.2831,0.2787,0.3110,0.2993,0.2812,0.2862
oil,0.1923,0.1634,0.1532,0.1716,0.1725,0.2052,0.2085,0.1541,0.1631
gold,0.1010,0.0834,0.0773,0.0845,0.0797,0.0878,0.0833,0.0830,0.0764
silver,0.2607,0.2023,0.2029,0.2102,0.2077,0.2250,0.2184,0.2171,0.2015


Two extensions. The forest on log volatility nearly matches HARX for oil (0.1541 vs 0.1532). The hybrid is best for gold and silver, but by small margins (tested for significance in notebook 6).

In [12]:
pd.concat({"RF_log": rflog, "Hybrid": hyb}, axis=1).to_csv("hybrid_forecasts.csv")

Save.